<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/media/get_media_library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖼️ WordPress Media Image Fetcher

**Description:** Script untuk mengambil seluruh URL gambar beserta metadata-nya dari WordPress Media Library via REST API, lalu menyimpannya ke file JSON dan CSV.

---

## ✅ Expected Result

| Field | Contoh |
|---|---|
| `id` | `1042` |
| `title` | `bali-sunset-uluwatu` |
| `source_url` | `https://www.balitouristic.com/wp-content/uploads/2024/01/bali-sunset.jpg` |
| `url_filename` | `bali-sunset.jpg` |
| `mime_type` | `image/jpeg` |
| `width` × `height` | `1920 × 1080` |
| `alt_text` | `Sunset view at Uluwatu Temple Bali` |
| `size_names` | `thumbnail, medium, large, full` |
| `sizes_count` | `4` |
| `file_size` | `284500` |
| `post` | `88` |

Output disimpan ke:
- `wp_media_images.json`
- `wp_media_images.csv`

---

## 🔗 Links

| Resource | URL |
|---|---|
| 📓 Google Colab | [Buka Notebook](https://colab.research.google.com/drive/1g2V2_S4PSlD27kaRNcVt8kh2bcOr2jV8?usp=sharing) |
| 🤖 Claude Chat | [Buka Percakapan](https://claude.ai/chat/c21ae102-d05b-41f5-9e6e-d654bb425a13) |
| 📖 WP REST API Docs | [developer.wordpress.org/rest-api](https://developer.wordpress.org/rest-api/reference/media/) |

In [ ]:
# @title
"""
Script untuk mengambil semua image URL dan parameternya dari WordPress Media Library
Domain: https://www.balitouristic.com/
Kredential diambil dari Google Colab Secrets
"""

import requests
import json
import csv
import os
from urllib.parse import urlparse, parse_qs

# ============================================================
# RICH LOGGER SETUP
# ============================================================
try:
    from rich.console import Console
    from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn, MofNCompleteColumn
    from rich.table import Table
    from rich.panel import Panel
    from rich.text import Text
    from rich import box
    from rich.live import Live
    from rich.columns import Columns
    import rich
    RICH_AVAILABLE = True
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rich", "-q"])
    from rich.console import Console
    from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn, MofNCompleteColumn
    from rich.table import Table
    from rich.panel import Panel
    from rich.text import Text
    from rich import box
    from rich.live import Live
    from rich.columns import Columns
    RICH_AVAILABLE = True

console = Console()

def log_info(msg):    console.print(f"[bold cyan]ℹ[/bold cyan]  {msg}")
def log_success(msg): console.print(f"[bold green]✔[/bold green]  {msg}")
def log_warning(msg): console.print(f"[bold yellow]⚠[/bold yellow]  {msg}")
def log_error(msg):   console.print(f"[bold red]✘[/bold red]  {msg}")

# ============================================================
# AMBIL KREDENTIAL DARI GOOGLE COLAB SECRETS
# ============================================================
try:
    from google.colab import userdata
    WP_PASSWORD = userdata.get('pass_bali')
    WP_USERNAME = userdata.get('admin_bali')
    log_success("Kredential berhasil diambil dari Google Colab Secrets")
except Exception as e:
    log_warning(f"Gagal ambil dari Colab Secrets: {e}")
    WP_USERNAME = ""
    WP_PASSWORD = ""

# ============================================================
# KONFIGURASI
# ============================================================
BASE_URL    = "https://www.balitouristic.com/"
API_URL     = f"{BASE_URL}/wp-json/wp/v2/media"
PER_PAGE    = 100
OUTPUT_JSON = "wp_media_images.json"
OUTPUT_CSV  = "wp_media_images.csv"

# ============================================================
# FUNGSI UTAMA
# ============================================================

def fetch_all_media():
    """Ambil semua item media dari WordPress REST API dengan paginasi."""
    session = requests.Session()
    session.auth = (WP_USERNAME, WP_PASSWORD)

    all_media = []
    page = 1
    total_pages = None

    console.print()
    console.rule("[bold magenta]🌐 WordPress Media Fetcher[/bold magenta]")
    console.print(Panel(
        f"[bold]Endpoint :[/bold] [link={API_URL}]{API_URL}[/link]\n"
        f"[bold]User     :[/bold] {WP_USERNAME}\n"
        f"[bold]Per Page :[/bold] {PER_PAGE}",
        title="[bold blue]Konfigurasi",
        border_style="blue",
        padding=(0, 2),
    ))
    console.print()

    with Progress(
        SpinnerColumn(style="bold magenta"),
        TextColumn("[bold blue]{task.description}"),
        BarColumn(bar_width=40, style="magenta", complete_style="bold green"),
        MofNCompleteColumn(),
        TextColumn("halaman"),
        TimeElapsedColumn(),
        console=console,
        transient=False,
    ) as progress:
        task = progress.add_task("Fetching...", total=None)

        while True:
            params = {
                "per_page": PER_PAGE,
                "page": page,
                "media_type": "image",
            }

            try:
                response = session.get(API_URL, params=params, timeout=30)
            except requests.exceptions.RequestException as e:
                log_error(f"Request error: {e}")
                break

            if response.status_code == 400:
                progress.update(task, description="[green]Selesai!", completed=total_pages or page)
                break

            if response.status_code != 200:
                log_error(f"HTTP {response.status_code}: {response.text[:300]}")
                break

            data = response.json()
            if not data:
                break

            # Update total setelah halaman pertama
            if total_pages is None:
                total_pages = int(response.headers.get("X-WP-TotalPages", 1))
                total_items = int(response.headers.get("X-WP-Total", len(data)))
                progress.update(task, total=total_pages)
                log_info(f"Ditemukan [bold]{total_items}[/bold] gambar di [bold]{total_pages}[/bold] halaman")

            for item in data:
                parsed = parse_media_item(item)
                all_media.append(parsed)

            progress.update(
                task,
                advance=1,
                description=f"[bold blue]Halaman {page}/{total_pages}[/bold blue] — [green]{len(all_media)} gambar[/green]",
            )

            if page >= total_pages:
                progress.update(task, description="[bold green]✔ Semua halaman selesai diambil!")
                break

            page += 1

    return all_media


def parse_media_item(item):
    """Ekstrak field penting dari satu item media."""
    source_url  = item.get("source_url", "")
    parsed_url  = urlparse(source_url)
    media_details = item.get("media_details", {})

    sizes = {}
    for size_name, size_info in media_details.get("sizes", {}).items():
        sizes[size_name] = {
            "url"    : size_info.get("source_url", ""),
            "width"  : size_info.get("width", ""),
            "height" : size_info.get("height", ""),
            "file"   : size_info.get("file", ""),
            "mime"   : size_info.get("mime_type", ""),
        }

    return {
        "id"          : item.get("id"),
        "date"        : item.get("date"),
        "modified"    : item.get("modified"),
        "slug"        : item.get("slug"),
        "status"      : item.get("status"),
        "title"       : item.get("title", {}).get("rendered", ""),
        "author"      : item.get("author"),
        "source_url"  : source_url,
        "url_scheme"  : parsed_url.scheme,
        "url_netloc"  : parsed_url.netloc,
        "url_path"    : parsed_url.path,
        "url_filename": os.path.basename(parsed_url.path),
        "mime_type"   : item.get("mime_type"),
        "media_type"  : item.get("media_type"),
        "width"       : media_details.get("width"),
        "height"      : media_details.get("height"),
        "file_path"   : media_details.get("file"),
        "file_size"   : media_details.get("filesize"),
        "alt_text"    : item.get("alt_text", ""),
        "caption"     : item.get("caption", {}).get("rendered", ""),
        "description" : item.get("description", {}).get("rendered", ""),
        "link"        : item.get("link"),
        "guid"        : item.get("guid", {}).get("rendered", ""),
        "sizes_json"  : json.dumps(sizes, ensure_ascii=False),
        "sizes_count" : len(sizes),
        "size_names"  : ", ".join(sizes.keys()),
        "post"        : item.get("post"),
    }


def save_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    log_success(f"Disimpan ke JSON : [bold]{filename}[/bold]")


def save_csv(data, filename):
    if not data:
        log_warning("Tidak ada data untuk disimpan ke CSV.")
        return

    fieldnames = list(data[0].keys())
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    log_success(f"Disimpan ke CSV  : [bold]{filename}[/bold]")


def print_summary(media_list):
    """Tampilkan ringkasan statistik dan preview tabel."""
    console.print()
    console.rule("[bold magenta]📊 Ringkasan Hasil[/bold magenta]")

    # --- Stat cards ---
    total     = len(media_list)
    mimes     = {}
    for m in media_list:
        mt = m.get("mime_type") or "unknown"
        mimes[mt] = mimes.get(mt, 0) + 1

    mime_lines = "\n".join(f"  [cyan]{k}[/cyan]: [bold]{v}[/bold]" for k, v in sorted(mimes.items(), key=lambda x: -x[1]))
    stats_panel = Panel(
        f"[bold green]Total Gambar  : {total}[/bold green]\n\n"
        f"[bold]Format Breakdown:[/bold]\n{mime_lines}",
        title="[bold]Statistik",
        border_style="green",
        padding=(0, 2),
    )
    console.print(stats_panel)

    # --- Preview tabel ---
    console.print()
    console.print("[bold]🖼  Preview 5 Item Pertama:[/bold]")
    table = Table(
        box=box.ROUNDED,
        border_style="dim",
        header_style="bold magenta",
        show_lines=True,
    )
    table.add_column("ID",        style="cyan",  no_wrap=True)
    table.add_column("Filename",  style="white", max_width=35)
    table.add_column("Dimensi",   style="yellow", justify="center")
    table.add_column("MIME",      style="green",  justify="center")
    table.add_column("Sizes",     style="dim",    max_width=40)

    for item in media_list[:5]:
        w = str(item.get("width") or "?")
        h = str(item.get("height") or "?")
        table.add_row(
            str(item["id"]),
            item["url_filename"],
            f"{w}×{h}",
            item.get("mime_type") or "-",
            item.get("size_names") or "-",
        )

    console.print(table)
    console.print()


# ============================================================
# ENTRY POINT
# ============================================================
if __name__ == "__main__":
    if not WP_USERNAME or not WP_PASSWORD:
        console.print(Panel(
            "[bold red]Username / Password kosong![/bold red]\n\n"
            "Pastikan Google Colab Secrets sudah diset:\n"
            "  • [cyan]pass_bali[/cyan]  → password WordPress\n"
            "  • [cyan]admin_bali[/cyan] → username WordPress",
            title="[bold red]❌ Error",
            border_style="red",
        ))
    else:
        media_list = fetch_all_media()

        if media_list:
            console.print()
            save_json(media_list, OUTPUT_JSON)
            save_csv(media_list, OUTPUT_CSV)
            print_summary(media_list)
            console.print(Panel(
                f"[bold green]✔ Selesai![/bold green] {len(media_list)} gambar berhasil diambil dan disimpan.",
                border_style="green",
            ))
        else:
            log_warning("Tidak ada gambar yang berhasil diambil. Periksa kredential / koneksi.")